# 05 — Script Generation

Turn one saved `VideoOutline` into complete, structured narration for an
educational short video.

This notebook loads an outline, generates narration, normalizes timing from
actual word count, saves the validated script, and previews the result.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.outlines import load_outline
from educational_shorts.prompts import load_prompt
from educational_shorts.scripts import (
    build_script_filename,
    find_outline_file,
    generate_script,
    save_script,
)

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

In [2]:
OUTLINES_DIRECTORY = PROJECT_ROOT / "data" / "outlines"
SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "scripts"

# Set a specific filename, or leave as None to use the newest outline.
OUTLINE_FILENAME = None

TARGET_WORDS_PER_MINUTE = 145
TEMPERATURE = 0.5
GENERATION_SEED = 42

print(f"Outlines directory: {OUTLINES_DIRECTORY}")
print(f"Scripts directory: {SCRIPTS_DIRECTORY}")

Outlines directory: c:\Users\hitch\python_files\educational_shorts\data\outlines
Scripts directory: c:\Users\hitch\python_files\educational_shorts\data\scripts


## Load an outline

In [3]:
outline_path = find_outline_file(
    outlines_directory=OUTLINES_DIRECTORY,
    filename=OUTLINE_FILENAME,
)

video_outline = load_outline(outline_path)

print(f"Loaded outline from: {outline_path}")
print(f"Topic: {video_outline.topic.title}")
print(
    f"Outline target: {video_outline.estimated_total_seconds} seconds "
    f"across {len(video_outline.sections)} body sections"
)

Loaded outline from: c:\Users\hitch\python_files\educational_shorts\data\outlines\how_do_bacteria_communicate.json
Topic: How Do Bacteria Communicate?
Outline target: 60 seconds across 4 body sections


## Load the script-generation prompt

In [4]:
script_system_prompt = load_prompt("script_generation")
print("Script-generation prompt loaded.")

Script-generation prompt loaded.


## Generate the script

In [5]:
video_script = generate_script(
    outline=video_outline,
    system_prompt=script_system_prompt,
    target_wpm=TARGET_WORDS_PER_MINUTE,
    temperature=TEMPERATURE,
    seed=GENERATION_SEED,
)

print(f"Generated {video_script.word_count} words.")
print(
    f"Estimated spoken duration: "
    f"{video_script.estimated_total_seconds} seconds"
)

Generated 135 words.
Estimated spoken duration: 56 seconds


## Save the script

In [6]:
output_path = SCRIPTS_DIRECTORY / build_script_filename(video_outline)

save_script(video_script, output_path)
print(f"Saved script to {output_path}")

Saved script to c:\Users\hitch\python_files\educational_shorts\data\scripts\how_do_bacteria_communicate.json


## Preview structured segments

In [7]:
print(f"TITLE: {video_script.topic.title}")
print()

print(f"HOOK ({video_script.hook.estimated_seconds}s)")
print(video_script.hook.narration)
print(f"Visual: {video_script.hook.visual_direction}")
print()

for index, section in enumerate(video_script.sections, start=1):
    print(
        f"{index}. {section.segment_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(section.narration)
    print(f"Visual: {section.visual_direction}")
    print()

print(f"CLOSING ({video_script.closing.estimated_seconds}s)")
print(video_script.closing.narration)
print(f"Visual: {video_script.closing.visual_direction}")
print()

print(f"WORD COUNT: {video_script.word_count}")
print(f"ESTIMATED TOTAL: {video_script.estimated_total_seconds} seconds")

TITLE: How Do Bacteria Communicate?

HOOK (10s)
Did you know that bacteria can talk to each other? They don’t use words, but they do have a secret language made of chemicals.
Visual: 

1. INTRODUCTION TO BACTERIAL COMMUNICATION (8s)
Bacteria are not solitary; they interact with each other. They use chemical signals called quorum sensing to coordinate their actions.
Visual: Show a simple animation of bacteria in a colony, with glowing dots representing chemical signals.

2. WHAT IS QUORUM SENSING? (9s)
Bacteria release molecules into their environment. These molecules act as messages to other bacteria, helping them know when they’re in a crowd.
Visual: Use a diagram showing bacteria releasing signal molecules, with arrows indicating communication pathways.

3. EXAMPLES OF BACTERIAL COMMUNICATION (11s)
Bacteria coordinate to form biofilms, which are sticky communities that protect them. They can also switch from harmless to harmful states when they sense enough neighbors.
Visual: Show a 

## Preview complete narration

In [8]:
print(video_script.full_narration)

Did you know that bacteria can talk to each other? They don’t use words, but they do have a secret language made of chemicals.

Bacteria are not solitary; they interact with each other. They use chemical signals called quorum sensing to coordinate their actions.

Bacteria release molecules into their environment. These molecules act as messages to other bacteria, helping them know when they’re in a crowd.

Bacteria coordinate to form biofilms, which are sticky communities that protect them. They can also switch from harmless to harmful states when they sense enough neighbors.

Understanding bacterial communication helps fight infections. It’s also a key area of microbiology research that impacts medicine and public health.

Bacteria use chemical signals to 'talk' and work together, which is crucial for their survival and has major implications for science and health.
